# 02 — Modélisation du churn et suivi MLflow

**Entrées :** les CSV et le contrat créés par le notebook 01. **Sortie :** les modèles et leurs runs MLflow, consommés par le notebook 03.

Aucune lecture de la source brute, création de feature, estimation de bornes tarifaires ou copie de `ChurnFeatureEngineer` ici. Les conversions de matrice sont uniquement techniques : catégories en texte et numériques en flottants.

Parcours : imports et paramètres → données préparées vérifiées → configuration MLflow → helpers de suivi → entraînement principal dans son run → courbes et seuil → diagnostics séparés. Le test est archivé avec le principal, sans être scoré ici ; son évaluation et le bootstrap sont dans le 03.

## 1 — Imports

In [17]:
# Bibliothèque standard
import hashlib
import json
import os
import re
import sys
import tempfile
import time
from importlib.metadata import version
from pathlib import Path

# Désactiver la télémétrie avant l'import de MLflow.
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")

# Données et graphiques
# Suivi d'expériences
import mlflow
import mlflow.catboost
import mlflow.data
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Modélisation et métriques
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.display import display
from mlflow.data.sources import LocalArtifactDatasetSource
from mlflow.entities import Metric
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from plotly.subplots import make_subplots
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Configuration locale : même import depuis la racine ou notebooks/.
if not Path("settings.py").is_file() and Path("../settings.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from settings import settings

## 2 — Paramètres des modèles

Les fenêtres et colonnes proviennent du contrat du 01. Les réglages d’apprentissage sont définis ici.

In [18]:
SEED = 42
np.random.seed(SEED)

# Paramètres communs aux entraînements et ablations comparables.
MODEL_PARAMS = dict(
    iterations=1800,
    depth=6,
    learning_rate=0.035,
    loss_function="Logloss",
    eval_metric="PRAUC:hints=skip_train~false",
    l2_leaf_reg=5,
    random_seed=SEED,
    allow_writing_files=False,
    thread_count=-1,
)
EARLY_STOPPING_ROUNDS = 120
LOG_EVERY = 50
DIAGNOSTIC_ITERATIONS = 400


NOTEBOOK_PATH = settings.PROJECT_ROOT / "notebooks" / "02_churn_modelisation.ipynb"
PACKAGE_NAMES = ["mlflow", "catboost", "numpy", "pandas", "scikit-learn", "plotly", "pydantic"]
FEATURE_CONTRACT_PATH = (
    settings.PROJECT_ROOT / "data" / "interim" / "features" / "feature_contract.json"
)

## 3 — Charger les données préparées

Le manifeste du 01 fournit les colonnes et métadonnées, puis chaque CSV est vérifié par SHA-256 avant lecture. Une préparation absente ou incohérente demande de relancer le 01 ; aucun feature engineering de secours n'est exécuté.

In [19]:
if not FEATURE_CONTRACT_PATH.is_file():
    raise FileNotFoundError("Exécuter le notebook 01 jusqu'à l'écriture du contrat de features.")
feature_contract = json.loads(FEATURE_CONTRACT_PATH.read_text())
if feature_contract.get("preparation_schema_version") != 1:
    raise ValueError("Version de contrat de préparation non prise en charge.")
SEED = feature_contract["seed"]
MODEL_PARAMS["random_seed"] = SEED
MODEL_FEATURES = feature_contract["model_features"]
ORIGINAL_FEATURES = feature_contract["original_features"]
NEW_FEATURES = feature_contract["new_features"]
CATEGORICAL_FEATURES = feature_contract["categorical_features"]
EXCLUDED = feature_contract["excluded"]
feature_metadata = feature_contract["feature_metadata"]


def verified_preparation_path(entry):
    path = settings.PROJECT_ROOT / entry["path"]
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != entry["sha256"]:
        raise ValueError(f"Préparation absente ou modifiée : {path}. Relancer le notebook 01.")
    return path


def load_prepared_frame(entry):
    path = verified_preparation_path(entry)
    frame = pd.read_csv(
        path,
        dtype={"customer_id": "string", "dept": "string"},
        parse_dates=["snapshot_date", "signup_date"],
    )
    missing = set(MODEL_FEATURES + ["customer_id", "snapshot_date", "churn"]) - set(frame.columns)
    if missing or len(frame) != entry["rows"]:
        raise ValueError(f"Schéma ou nombre de lignes incorrect dans {path} : {sorted(missing)}")
    if (
        frame.empty
        or frame.churn.isna().any()
        or not frame.churn.isin([0, 1]).all()
        or frame.churn.nunique() != 2
    ):
        raise ValueError(f"Deux classes binaires non manquantes requises : {path}")
    if (
        frame.customer_id.isna().any()
        or frame.snapshot_date.isna().any()
        or frame.duplicated(["customer_id", "snapshot_date"]).any()
    ):
        raise ValueError(f"Clé client × snapshot invalide : {path}")
    return frame


train_df = load_prepared_frame(feature_contract["datasets"]["train"])
val_df = load_prepared_frame(feature_contract["datasets"]["validation"])
test_df = load_prepared_frame(feature_contract["datasets"]["test"])
assert train_df.snapshot_date.max() < val_df.snapshot_date.min()
assert val_df.snapshot_date.max() < test_df.snapshot_date.min()
verified_preparation_path(feature_contract["source"])
preparation_reports = {
    name: pd.read_csv(verified_preparation_path(entry))
    for name, entry in feature_contract["reports"].items()
}
print("Préparation chargée :", feature_contract["preparation_id"])
print("Train / validation / test :", len(train_df), len(val_df), len(test_df))
display(
    pd.DataFrame(
        {
            "feature": MODEL_FEATURES,
            "catégorielle": [c in CATEGORICAL_FEATURES for c in MODEL_FEATURES],
        }
    )
)

Préparation chargée : d20df3e04b951dabd9f2f0cba960355eb0b2e951bc0a871f49b20973a64ee40d
Train / validation / test : 23322 6309 6091


,feature,catégorielle
0,contract,True
1,region,True
2,dept,True
3,channel,True
4,monthly_fee,False
5,data_gb,False
6,support_calls_3m,False
7,late_payments,False
8,nps,False
9,last_invoice_amount,False


### 3.1 — Conversions techniques pour CatBoost

Les features existent déjà dans les CSV ; aucun calcul métier ni apprentissage dans cette cellule.

In [20]:
def make_matrix(part: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    x = part[features].copy()

    for column in features:
        if column in CATEGORICAL_FEATURES:
            x[column] = x[column].astype("string").fillna("__MISSING__").astype(str)
        else:
            x[column] = pd.to_numeric(x[column], errors="coerce").astype(float)
            x[column] = x[column].replace([np.inf, -np.inf], np.nan)

    return x


X_train = make_matrix(train_df, MODEL_FEATURES)
X_val = make_matrix(val_df, MODEL_FEATURES)

y_train = train_df["churn"].to_numpy()
y_val = val_df["churn"].to_numpy()

print("Train :", X_train.shape)
print("Validation :", X_val.shape)

Train : (23322, 30)
Validation : (6309, 30)


## 4 — Configuration MLflow locale

Configurer une seule fois le stockage et l’expérience. Aucun run n’est encore ouvert ; le bloc d’entraînement de la section 6 gère son propre cycle de vie.

In [21]:
settings.MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(settings.MLFLOW_EXPERIMENT)
if experiment is None:
    experiment_id = mlflow.create_experiment(
        settings.MLFLOW_EXPERIMENT,
        artifact_location=(settings.MLFLOW_DIR / "artifacts").as_uri(),
    )
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print("MLflow :", mlflow.__version__)
print("Expérience :", settings.MLFLOW_EXPERIMENT)
print("Stockage local :", settings.MLFLOW_DIR)

MLflow : 3.18.0
Expérience : telco-churn
Stockage local : /Users/surelmanda/Downloads/sujet-entretien-technique/Telecom-Retention-Optimize/.mlflow


## 5 — Fonctions de suivi MLflow

Ces fonctions ne lancent aucun entraînement et n'ouvrent aucun run. Elles sont utilisées **à l'intérieur** des blocs `with mlflow.start_run(...)` de la section 6 et des diagnostics.

| Fonction | Responsabilité | Où voir le résultat |
|---|---|---|
| `log_training_source` | Notebook sans sorties, export Python, cellule d'entraînement, configurations et environnement | Artifacts → `source/`, `environment.json` |
| `log_training_inputs` | Paramètres, instantanés CSV, SHA-256 et filiation des datasets | Parameters, Datasets, Artifacts → `data/` |
| `MLflowTrainingCallback` | Métriques pendant l'apprentissage | Metrics → `learning/...` |
| `log_training_results` | Scores finaux, historique complet et contrat du modèle | Metrics, Artifacts → `learning/`, `contract.json` |

L'appel natif `mlflow.catboost.log_model` reste visible dans l'entraînement principal. Le helper `log_diagnostic_model` applique la même sauvegarde aux diagnostics pour éviter de répéter ce code.

API suivies : [Tracking](https://mlflow.org/docs/latest/ml/tracking/tracking-api/), [Datasets](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.data.html), [CatBoost](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.catboost.html).

### 5.1 — Traçabilité du code et de l’environnement

Le notebook et l’export `.py` sont des instantanés du fichier sur disque. Dans Jupyter, `training_cell.py` capture aussi la cellule exécutée, même si elle n’est pas encore sauvegardée ; hors Jupyter, il utilise la cellule identifiée dans le fichier. L’origine est enregistrée dans `source/provenance.json`. Sauvegarder le notebook avant l’entraînement reste nécessaire pour figer toutes les définitions amont.

In [22]:
def training_tags(role, parent_run_id=None):
    tags = {"stage": "training", "role": role, "contract_version": "1"}
    if parent_run_id is not None:
        tags["parent_training_run_id"] = parent_run_id
        tags["mlflow.parentRunId"] = parent_run_id
    return tags


def require_no_active_run():
    if mlflow.active_run() is not None:
        raise RuntimeError("Terminer le run MLflow actif avant de lancer cet entraînement.")


def log_training_source(cell_id):
    """Archiver le notebook sur disque et la cellule réellement exécutée si disponible."""
    snapshot = json.loads(NOTEBOOK_PATH.read_text())
    code_cells = [cell for cell in snapshot["cells"] if cell["cell_type"] == "code"]
    for cell in code_cells:
        cell["outputs"] = []
        cell["execution_count"] = None
    # Les sorties du notebook ne sont pas nécessaires pour reproduire le code.
    mlflow.log_text(json.dumps(snapshot, ensure_ascii=False, indent=1), "source/notebook_02.ipynb")
    exported_code = "\n\n".join("".join(cell["source"]) for cell in code_cells)
    mlflow.log_text(exported_code, "source/notebook_02.py")
    disk_cell = next(cell for cell in code_cells if cell["id"] == cell_id)
    executed_code = "".join(disk_cell["source"])
    origin = "cellule_du_notebook_sur_disque"
    shell = get_ipython()
    if shell is not None and shell.history_manager.input_hist_raw[-1].strip():
        executed_code = shell.history_manager.input_hist_raw[-1]
        origin = "cellule_executee_dans_le_noyau"
    mlflow.log_text(executed_code, "source/training_cell.py")
    mlflow.log_dict(
        {
            "training_cell_id": cell_id,
            "training_cell_source": origin,
            "notebook_disk_sha256": hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest(),
        },
        "source/provenance.json",
    )
    for filename in ["settings.py", "pyproject.toml", "poetry.lock"]:
        path = settings.PROJECT_ROOT / filename
        if path.is_file():
            mlflow.log_artifact(str(path), artifact_path="source/configuration")
    feature_path = verified_preparation_path(feature_contract["source"])
    mlflow.log_artifact(str(feature_path), artifact_path="source/features")
    mlflow.log_dict(feature_contract, "source/features/feature_contract.json")
    mlflow.log_dict(
        {
            "python": sys.version,
            "packages": {package: version(package) for package in PACKAGE_NAMES},
        },
        "environment.json",
    )

### 5.2 — Paramètres, snapshots et filiation des données

Les CSV exacts sont archivés et reliés à l’onglet Datasets avec leur contexte. Le test est marqué `test_reference` : il est archivé, mais jamais utilisé dans `fit` ou dans la sélection du seuil.

In [23]:
def log_training_inputs(estimator, frames, features, feature_metadata, cell_id):
    """Logguer paramètres, code et données avant fit, dans le run déjà ouvert."""
    active_run = mlflow.active_run()
    if active_run is None:
        raise RuntimeError("Ouvrir un run avec mlflow.start_run avant de logguer.")
    mlflow.log_params(estimator.get_params())
    mlflow.log_param("preparation_id", feature_contract["preparation_id"])
    mlflow.log_params(
        {
            "feature_count": len(features),
            "train_rows": len(frames["train"]),
            "validation_rows": len(frames["validation"]),
        }
    )
    contract = {
        "contract_version": 1,
        "model_features": list(features),
        "categorical_features": [c for c in CATEGORICAL_FEATURES if c in features],
        "new_features": list(NEW_FEATURES),
        "excluded": list(EXCLUDED),
        "threshold": None,
        "seed": SEED,
        "train_rows": len(frames["train"]),
        "validation_rows": len(frames["validation"]),
        "matrix_preparation": "categorical_string_missing__MISSING__;numeric_float_nan",
        "datasets": {},
        **feature_metadata,
        "preparation_id": feature_contract["preparation_id"],
    }

    logged_datasets = {}
    with tempfile.TemporaryDirectory(prefix="telco-training-inputs-") as temporary:
        for label, frame in frames.items():
            path = Path(temporary) / f"{label}.csv"
            frame.to_csv(path, index=False)
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            mlflow.log_artifact(str(path), artifact_path="data")
            contract["datasets"][label] = {
                "artifact_path": f"data/{label}.csv",
                "rows": len(frame),
                "sha256": digest,
            }
            # Filiation visible dans l'onglet Datasets de MLflow.
            # Le test est une référence archivée, jamais une entrée de fit.
            context = {"train": "training", "validation": "validation", "test": "test_reference"}[
                label
            ]
            dataset_frame = make_matrix(frame, features)
            # Copie flottante pour le schéma MLflow ; cible et CSV archivés inchangés.
            dataset_frame["churn"] = frame.churn.to_numpy(dtype=float)
            dataset = mlflow.data.from_pandas(
                dataset_frame,
                source=LocalArtifactDatasetSource(
                    f"{active_run.info.artifact_uri}/data/{label}.csv"
                ),
                name=label,
                targets="churn",
            )
            mlflow.log_input(dataset, context=context)
            logged_datasets[label] = dataset
    log_training_source(cell_id)
    return contract, logged_datasets

### 5.3 — Métriques d’apprentissage et sauvegarde finale

Le callback publie la première itération puis chaque `LOG_EVERY` itérations. À la fin, les points restants complètent l’historique sans duplication. Toutes les itérations sont aussi disponibles en CSV, JSON et graphique HTML. Les métriques finales sont associées au modèle et au dataset correspondants.

In [24]:
def learning_metric_name(split, metric):
    return re.sub(r"[^\w./-]+", "_", f"learning/{split}/{metric}").lower()


class MLflowTrainingCallback:
    """Publier l'avancement pendant fit, à la première itération puis toutes les LOG_EVERY itérations."""

    def after_iteration(self, info):
        if info.iteration == 1 or info.iteration % LOG_EVERY == 0:
            values = {
                learning_metric_name(split, metric): float(history[-1])
                for split, metrics in info.metrics.items()
                for metric, history in metrics.items()
                if history and np.isfinite(history[-1])
            }
            mlflow.log_metrics(values, step=info.iteration)
        return True


def log_learning_history(estimator):
    """Compléter toutes les itérations et garder un CSV facilement consultable."""
    history = estimator.get_evals_result()
    mlflow.log_dict(history, "learning/history.json")
    rows, remaining_metrics = [], []
    timestamp = int(time.time() * 1000)
    for split, metrics in history.items():
        for metric, values in metrics.items():
            for iteration, value in enumerate(values, start=1):
                if not np.isfinite(value):
                    continue
                key = learning_metric_name(split, metric)
                rows.append(
                    {"split": split, "metric": metric, "iteration": iteration, "value": value}
                )
                # Les points déjà envoyés par le callback ne sont pas dupliqués.
                if iteration != 1 and iteration % LOG_EVERY != 0:
                    remaining_metrics.append(Metric(key, float(value), timestamp, iteration))
    client = MlflowClient()
    run_id = mlflow.active_run().info.run_id
    for start in range(0, len(remaining_metrics), 1000):
        client.log_batch(run_id, metrics=remaining_metrics[start : start + 1000])
    mlflow.log_text(pd.DataFrame(rows).to_csv(index=False), "learning/history.csv")
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Logloss", "PRAUC"))
    for split, color in [("learn", "#2563EB"), ("validation", "#E11D48")]:
        for column, metric in [(1, "Logloss"), (2, "PRAUC")]:
            values = history.get(split, {}).get(metric, [])
            if values:
                fig.add_trace(
                    go.Scatter(
                        x=list(range(1, len(values) + 1)),
                        y=values,
                        name=split,
                        legendgroup=split,
                        showlegend=column == 1,
                        line=dict(color=color),
                    ),
                    row=1,
                    col=column,
                )
    fig.update_layout(template="plotly_white", title="Historique d'apprentissage")
    mlflow.log_figure(fig, "learning/curves.html")


def log_training_results(estimator, contract, model_info, frames, datasets, threshold=None):
    """Finaliser le même run ; aucun start_run caché dans cette fonction."""
    mlflow.log_dict(estimator.get_all_params(), "model/effective_parameters.json")
    contract["model_uri"] = model_info.model_uri
    contract["threshold"] = threshold
    mlflow.log_params(
        {
            "fitted_tree_count": estimator.tree_count_,
            "best_iteration": estimator.get_best_iteration(),
        }
    )
    if threshold is not None:
        mlflow.log_param("decision_threshold", threshold)
    for label in ["train", "validation"]:
        frame = frames[label]
        probability = estimator.predict_proba(make_matrix(frame, contract["model_features"]))[:, 1]
        target = frame.churn.to_numpy(dtype=int)
        scores = {
            "average_precision": average_precision_score(target, probability),
            "roc_auc": roc_auc_score(target, probability),
            "log_loss": log_loss(target, probability, labels=[0, 1]),
            "brier": brier_score_loss(target, probability),
        }
        if threshold is not None:
            prediction = (probability >= threshold).astype(int)
            scores.update(
                {
                    "mcc": matthews_corrcoef(target, prediction),
                    "precision": precision_score(target, prediction, zero_division=0),
                    "recall": recall_score(target, prediction, zero_division=0),
                    "f1": f1_score(target, prediction, zero_division=0),
                }
            )
        mlflow.log_metrics(
            {f"{label}_{key}": float(value) for key, value in scores.items()},
            model_id=model_info.model_id,
            dataset=datasets[label],
        )
    log_learning_history(estimator)
    mlflow.log_dict(contract, "contract.json")
    # Écrit en dernier : le notebook 02 ne prend que les runs FINISHED avec ce rôle.
    mlflow.set_tag("training_artifacts_complete", "true")


def log_diagnostic_model(estimator, contract, frames, datasets):
    """Même sauvegarde native que pour le principal ; aucun nouvel entraînement."""
    matrix = make_matrix(frames["train"], contract["model_features"])
    model_info = mlflow.catboost.log_model(
        estimator,
        name="catboost",
        signature=infer_signature(matrix, estimator.predict(matrix)),
        input_example=matrix.head(5),
        pip_requirements=[
            f"{package}=={version(package)}"
            for package in ["catboost", "pandas", "numpy", "scikit-learn", "mlflow"]
        ],
    )
    log_training_results(estimator, contract, model_info, frames, datasets)
    return model_info

## 6 — Entraîner le modèle principal dans un run MLflow

Les données et le code sont archivés avant `fit`. Le seuil est choisi sur validation, puis le modèle natif et son contrat sont sauvegardés avant la fermeture du run. En cas d'exception, le run est marqué `FAILED`. Le 03 ne sélectionne que le dernier principal terminé avec succès.

In [25]:
require_no_active_run()

# Le run commence AVANT fit et se termine après toutes les sauvegardes.
# En cas d'exception dans ce bloc, MLflow marque le run FAILED.
with mlflow.start_run(run_name="catboost_principal", tags=training_tags("principal")) as run:
    training_run_id = run.info.run_id
    print("Run principal :", training_run_id)

    # 1. Créer le modèle et archiver paramètres, données, code et environnement.
    model = CatBoostClassifier(**MODEL_PARAMS, custom_metric=["Logloss", "AUC"], verbose=LOG_EVERY)
    training_frames = {"train": train_df, "validation": val_df, "test": test_df}
    training_contract, training_datasets = log_training_inputs(
        model,
        training_frames,
        MODEL_FEATURES,
        feature_metadata,
        cell_id="train-principal",
    )
    for name, table in preparation_reports.items():
        mlflow.log_text(table.to_csv(index=False), f"preparation/{name}.csv")
    mlflow.log_dict(feature_contract["quality"], "preparation/quality_summary.json")
    mlflow.log_params({"early_stopping_rounds": EARLY_STOPPING_ROUNDS, "use_best_model": True})

    # 2. Entraîner : le callback envoie les métriques dans le run en cours.
    started_at = time.perf_counter()
    model.fit(
        X_train,
        y_train,
        cat_features=[c for c in CATEGORICAL_FEATURES if c in X_train.columns],
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        use_best_model=True,
        verbose=LOG_EVERY,
        callbacks=[MLflowTrainingCallback()],
    )
    mlflow.log_metric("fit_duration_seconds", time.perf_counter() - started_at)

    # 3. Choisir le seuil uniquement sur validation, toujours dans le même run.
    validation_probability = model.predict_proba(X_val)[:, 1]
    threshold_grid = np.linspace(0.05, 0.95, 181)
    threshold_scores = pd.DataFrame(
        {
            "threshold": threshold_grid,
            "MCC": [
                matthews_corrcoef(y_val, (validation_probability >= t).astype(int))
                for t in threshold_grid
            ],
            "F1": [
                f1_score(y_val, (validation_probability >= t).astype(int), zero_division=0)
                for t in threshold_grid
            ],
        }
    )
    best_threshold = float(threshold_scores.loc[threshold_scores.MCC.idxmax(), "threshold"])
    mlflow.log_text(threshold_scores.to_csv(index=False), "selection/threshold_scores.csv")
    threshold_figure = px.line(
        threshold_scores, x="threshold", y=["MCC", "F1"], title="Choix du seuil sur validation"
    )
    threshold_figure.add_vline(x=best_threshold, line_dash="dash", line_color="#E11D48")
    threshold_figure.update_layout(template="plotly_white", height=410)
    mlflow.log_figure(threshold_figure, "selection/threshold_curve.html")

    # 4. Sauvegarder le modèle au format CatBoost natif, avec signature et exemple.
    model_info = mlflow.catboost.log_model(
        model,
        name="catboost",
        signature=infer_signature(X_train, model.predict(X_train)),
        input_example=X_train.head(5),
        pip_requirements=[
            f"{package}=={version(package)}"
            for package in ["catboost", "pandas", "numpy", "scikit-learn", "mlflow"]
        ],
    )

    # 5. Sauvegarder les résultats, courbes par itération et contrat pour le 03.
    log_training_results(
        model,
        training_contract,
        model_info,
        training_frames,
        training_datasets,
        threshold=best_threshold,
    )

print("Run terminé :", training_run_id)
print("Modèle sauvegardé :", model_info.model_uri)
print("Meilleure itération :", model.best_iteration_)
print(f"Seuil de validation : {best_threshold:.3f}")

Run principal : 7b12e0c23caa4922b150fbbc69a30bc9
0:	learn: 0.9613593	test: 0.9545581	best: 0.9545581 (0)	total: 88.8ms	remaining: 2m 39s
50:	learn: 0.9962418	test: 0.9929638	best: 0.9931436 (27)	total: 1.28s	remaining: 44s
100:	learn: 0.9975500	test: 0.9928661	best: 0.9933177 (55)	total: 2.38s	remaining: 40.1s
150:	learn: 0.9984121	test: 0.9930110	best: 0.9933177 (55)	total: 3.45s	remaining: 37.7s
Stopped by overfitting detector  (120 iterations wait)

bestTest = 0.9933177447
bestIteration = 55

Shrink model to first 56 iterations.
Run terminé : 7b12e0c23caa4922b150fbbc69a30bc9
Modèle sauvegardé : models:/m-e5ce40243b2441efa0c3476f03f34557
Meilleure itération : 55
Seuil de validation : 0.370


## 7 — Lire les courbes d’apprentissage

In [26]:
history = model.get_evals_result()
print({split: list(metrics) for split, metrics in history.items()})

fig = make_subplots(rows=1, cols=2, subplot_titles=("Logloss", "PRAUC CatBoost"))

for name, color in [("learn", "#2563EB"), ("validation", "#E11D48")]:
    if name not in history:
        continue
    for col, metric in [(1, "Logloss"), (2, "PRAUC")]:
        if metric in history[name]:
            fig.add_trace(
                go.Scatter(
                    y=history[name][metric],
                    mode="lines",
                    name="Train" if name == "learn" else "Validation",
                    legendgroup=name,
                    showlegend=(col == 1),
                    line=dict(color=color, width=2),
                ),
                row=1,
                col=col,
            )

fig.update_layout(
    template="plotly_white", title="Évolution des métriques pendant l'entraînement", height=450
)
fig.update_xaxes(title_text="Itérations")
fig.show(config={"responsive": True})

{'learn': ['Logloss', 'PRAUC'], 'validation': ['Logloss', 'PRAUC', 'AUC']}


### 7.1 — Relire le seuil sauvegardé

Le choix a été fait dans le run précédent ; cette cellule l’affiche sans le recalculer.

In [27]:
probas = {"Train": model.predict_proba(X_train)[:, 1], "Validation": validation_probability}
print(f"Seuil enregistré : {best_threshold:.3f}")
threshold_figure.show(config={"responsive": True})

Seuil enregistré : 0.370


## 8 — Diagnostics : modèles séparés du principal

Les jeux de données de chaque protocole ont été préparés dans le 01. Chaque entraînement porte le rôle `diagnostic`, avec un lien vers le principal ; il ne le remplace pas dans le 03.

### 8.1 — Ablation des variables créées

Nous comparons deux modèles entraînés avec les **mêmes mois**, les **mêmes hyperparamètres** et la **même validation**. Les résultats de test du modèle enrichi ne doivent pas être utilisés pour choisir les features. Le baseline n'utilise que les variables d'origine autorisées.

Une analyse supplémentaire de la famille facturation peut être envisagée après confirmation de la temporalité des données.

In [28]:
X_train_base = make_matrix(train_df, ORIGINAL_FEATURES)
X_val_base = make_matrix(val_df, ORIGINAL_FEATURES)

require_no_active_run()
with mlflow.start_run(
    run_name="ablation_originales", tags=training_tags("diagnostic", training_run_id)
) as diagnostic_run:
    baseline = CatBoostClassifier(**MODEL_PARAMS, verbose=LOG_EVERY)

    diagnostic_frames = {"train": train_df, "validation": val_df}
    diagnostic_contract, diagnostic_datasets = log_training_inputs(
        baseline,
        diagnostic_frames,
        ORIGINAL_FEATURES,
        feature_metadata,
        cell_id="train-baseline",
    )
    mlflow.log_params({"early_stopping_rounds": EARLY_STOPPING_ROUNDS, "use_best_model": True})

    baseline.fit(
        X_train_base,
        y_train,
        cat_features=[c for c in CATEGORICAL_FEATURES if c in X_train_base.columns],
        eval_set=(X_val_base, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        use_best_model=True,
        verbose=LOG_EVERY,
        callbacks=[MLflowTrainingCallback()],
    )

    log_diagnostic_model(baseline, diagnostic_contract, diagnostic_frames, diagnostic_datasets)
    baseline_run_id = diagnostic_run.info.run_id

base_val_proba = baseline.predict_proba(X_val_base)[:, 1]

comparison = pd.DataFrame(
    [
        {
            "model": "Originales seules",
            "features": len(ORIGINAL_FEATURES),
            "Average Precision": average_precision_score(y_val, base_val_proba),
            "ROC-AUC": roc_auc_score(y_val, base_val_proba),
            "Logloss": log_loss(y_val, base_val_proba),
        },
        {
            "model": "Originales + métiers retenues",
            "features": len(MODEL_FEATURES),
            "Average Precision": average_precision_score(y_val, probas["Validation"]),
            "ROC-AUC": roc_auc_score(y_val, probas["Validation"]),
            "Logloss": log_loss(y_val, probas["Validation"]),
        },
    ]
)

display(comparison)

fig = px.bar(
    comparison,
    x="model",
    y="Average Precision",
    color="model",
    text_auto=".3f",
    title="Ablation sur validation — Average Precision",
)
fig.update_layout(template="plotly_white", showlegend=False, height=400)
fig.show(config={"responsive": True})

0:	learn: 0.5518376	test: 0.5715186	best: 0.5715186 (0)	total: 33.4ms	remaining: 1m
50:	learn: 0.9302490	test: 0.9438190	best: 0.9438190 (50)	total: 1.18s	remaining: 40.3s
100:	learn: 0.9771822	test: 0.9673132	best: 0.9674627 (98)	total: 2.26s	remaining: 38s
150:	learn: 0.9871519	test: 0.9714493	best: 0.9714493 (150)	total: 3.03s	remaining: 33.1s
200:	learn: 0.9901806	test: 0.9735766	best: 0.9735766 (200)	total: 3.8s	remaining: 30.2s
250:	learn: 0.9928412	test: 0.9771504	best: 0.9771504 (250)	total: 4.62s	remaining: 28.5s
300:	learn: 0.9949228	test: 0.9781205	best: 0.9781205 (299)	total: 5.44s	remaining: 27.1s
350:	learn: 0.9962257	test: 0.9801626	best: 0.9801626 (350)	total: 6.26s	remaining: 25.9s
400:	learn: 0.9972102	test: 0.9804269	best: 0.9807322 (363)	total: 7.24s	remaining: 25.3s
450:	learn: 0.9977672	test: 0.9807138	best: 0.9808335 (443)	total: 7.86s	remaining: 23.5s
500:	learn: 0.9983397	test: 0.9806200	best: 0.9808736 (475)	total: 8.91s	remaining: 23.1s
550:	learn: 0.9986120	

,model,features,Average Precision,ROC-AUC,Logloss
0,Originales seules,10,0.980898,0.996914,0.013365
1,Originales + métiers retenues,30,0.993330,0.999705,0.007385


### 8.2 — Clients jamais vus

**Pourquoi le comparer ?** Ce protocole mesure la capacité à prédire sur des clients **jamais vus à l'entraînement**. Il empêche tout chevauchement d'identifiants, mais mélange les mois : il **ne représente pas** la prédiction en production à une date donnée. Nous le limitons aux observations **avant octobre** pour ne pas entraîner sur le test chronologique final. Ses scores ne sont donc pas directement comparables à ceux du test temporel, et ne servent pas à choisir le modèle final.

`GroupShuffleSplit` est utilisé **une seule fois** comme diagnostic de sensibilité. CatBoost reste utilisé directement, sans wrapper.

In [29]:
group_entry = feature_contract["diagnostics"]["clients"]
group_train = load_prepared_frame(group_entry["datasets"]["train"])
group_holdout = load_prepared_frame(group_entry["datasets"]["validation"])
assert set(group_train.customer_id).isdisjoint(set(group_holdout.customer_id))
X_group_train = make_matrix(group_train, MODEL_FEATURES)
X_group_holdout = make_matrix(group_holdout, MODEL_FEATURES)
# Nombre d'arbres fixé a priori : aucun early stopping sur le holdout de diagnostic.
require_no_active_run()
with mlflow.start_run(
    run_name="diagnostic_clients", tags=training_tags("diagnostic", training_run_id)
) as diagnostic_run:
    group_model = CatBoostClassifier(
        iterations=DIAGNOSTIC_ITERATIONS,
        depth=6,
        learning_rate=0.035,
        loss_function="Logloss",
        random_seed=SEED,
        verbose=100,
        allow_writing_files=False,
        thread_count=-1,
    )
    diagnostic_frames = {"train": group_train, "validation": group_holdout}
    diagnostic_contract, diagnostic_datasets = log_training_inputs(
        group_model,
        diagnostic_frames,
        MODEL_FEATURES,
        group_entry["feature_metadata"],
        cell_id="train-group",
    )

    group_model.fit(
        X_group_train,
        group_train.churn.to_numpy(),
        cat_features=[c for c in CATEGORICAL_FEATURES if c in MODEL_FEATURES],
        verbose=100,
        callbacks=[MLflowTrainingCallback()],
    )
    log_diagnostic_model(group_model, diagnostic_contract, diagnostic_frames, diagnostic_datasets)
    group_run_id = diagnostic_run.info.run_id

p_group = group_model.predict_proba(X_group_holdout)[:, 1]
group_diagnostic = pd.DataFrame(
    [
        {
            "Protocole": "GroupShuffleSplit (clients jamais vus, mois mélangés)",
            "Snapshots": len(group_holdout),
            "Prévalence": group_holdout.churn.mean(),
            "Average Precision": average_precision_score(group_holdout.churn, p_group),
            "ROC-AUC": roc_auc_score(group_holdout.churn, p_group),
            "Brier": brier_score_loss(group_holdout.churn, p_group),
        }
    ]
)
display(group_diagnostic)

0:	learn: 0.5825866	total: 35.1ms	remaining: 14s
100:	learn: 0.0031844	total: 2.26s	remaining: 6.7s
200:	learn: 0.0021593	total: 5.03s	remaining: 4.98s
300:	learn: 0.0016360	total: 7.89s	remaining: 2.6s
399:	learn: 0.0013158	total: 10s	remaining: 0us


,Protocole,Snapshots,Prévalence,Average Precision,ROC-AUC,Brier
0,"GroupShuffleSplit (clients jamais vus, mois mé...",5913,0.030103,0.983642,0.999039,0.002032


### 8.3 — Walk-forward temporel

Pour prédire un snapshot du mois **M**, ce diagnostic entraîne sur les snapshots **jusqu'à M−2 inclus**, en excluant M−1 par prudence. Cela évite d'utiliser les labels du mois immédiatement précédent, qui ne seraient pas encore disponibles au moment du scoring de fin de mois. Cette règle est volontairement conservatrice : la disponibilité exacte dépend de l'heure de scoring et de la date réelle de réception du label.

Les folds août, septembre, octobre et novembre sont évalués séparément. **Aucun early stopping ni réglage de seuil n'est effectué sur le mois évalué**. Les hyperparamètres et les 400 itérations sont fixés avant ce diagnostic. Il ne s'agit pas d'une validation imbriquée complète ; une mise en production nécessiterait de sélectionner les hyperparamètres uniquement avec des fenêtres historiques antérieures à chaque scoring.

In [30]:
walk_rows = []
walk_predictions = []
for fold_entry in feature_contract["diagnostics"]["walk_forward"]:
    test_month = fold_entry["month"]
    last_train_month = fold_entry["last_train_month"]
    fold_train = load_prepared_frame(fold_entry["datasets"]["train"])
    fold_test = load_prepared_frame(fold_entry["datasets"]["validation"])
    assert fold_train.snapshot_date.max() < fold_test.snapshot_date.min()
    require_no_active_run()
    with mlflow.start_run(
        run_name=f"walk_forward_{test_month}", tags=training_tags("diagnostic", training_run_id)
    ) as diagnostic_run:
        fold_model = CatBoostClassifier(
            iterations=DIAGNOSTIC_ITERATIONS,
            depth=6,
            learning_rate=0.035,
            loss_function="Logloss",
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )
        diagnostic_frames = {"train": fold_train, "validation": fold_test}
        diagnostic_contract, diagnostic_datasets = log_training_inputs(
            fold_model,
            diagnostic_frames,
            MODEL_FEATURES,
            fold_entry["feature_metadata"],
            cell_id="train-walk-forward",
        )

        fold_model.fit(
            make_matrix(fold_train, MODEL_FEATURES),
            fold_train.churn.to_numpy(),
            cat_features=[c for c in CATEGORICAL_FEATURES if c in MODEL_FEATURES],
            verbose=False,
            callbacks=[MLflowTrainingCallback()],
        )
        log_diagnostic_model(
            fold_model, diagnostic_contract, diagnostic_frames, diagnostic_datasets
        )
        fold_run_id = diagnostic_run.info.run_id
    p = fold_model.predict_proba(make_matrix(fold_test, MODEL_FEATURES))[:, 1]
    walk_rows.append(
        {
            "Mois évalué": str(test_month),
            "Train jusqu'à": str(last_train_month),
            "Snapshots train": len(fold_train),
            "Snapshots test": len(fold_test),
            "Prévalence": fold_test.churn.mean(),
            "Average Precision": average_precision_score(fold_test.churn, p),
            "ROC-AUC": roc_auc_score(fold_test.churn, p),
            "Brier": brier_score_loss(fold_test.churn, p),
        }
    )
    walk_predictions.append(
        pd.DataFrame(
            {
                "month": str(test_month),
                "customer_id": fold_test.customer_id.to_numpy(),
                "y": fold_test.churn.to_numpy(),
                "p": p,
            }
        )
    )
walk_results = pd.DataFrame(walk_rows)
if walk_results.empty:
    raise ValueError("Aucun fold walk-forward valide")
display(walk_results)
walk_long = walk_results.melt(
    id_vars="Mois évalué",
    value_vars=["Average Precision", "ROC-AUC", "Brier"],
    var_name="Métrique",
    value_name="Valeur",
)
fig = px.line(
    walk_long,
    x="Mois évalué",
    y="Valeur",
    color="Métrique",
    markers=True,
    title="Stabilité mensuelle hors temps — validation walk-forward",
)
fig.update_layout(template="plotly_white", height=440)
fig.show(config={"responsive": True})

,Mois évalué,Train jusqu'à,Snapshots train,Snapshots test,Prévalence,Average Precision,ROC-AUC,Brier
0,2024-08,2024-06,20103,3183,0.035501,0.996596,0.999836,0.001163
1,2024-09,2024-07,23322,3126,0.030710,0.985088,0.999216,0.001664
2,2024-10,2024-08,26505,3088,0.027526,0.997863,0.999929,0.000738
3,2024-11,2024-09,29631,3003,0.031968,0.989857,0.999394,0.001077


### 8.4 — Ablation de la facture

Les fortes importances observées précédemment pour `invoice_fee_ratio`, `invoice_fee_gap` et `last_invoice_amount` ne prouvent **ni** une fuite **ni** un effet causal. Ces variables et leurs dérivées sont corrélées par construction. La variante « sans famille facture » retire aussi `payment_invoice_interaction` et `invoice_below_fee`. Nous évaluons leur contribution sur la **validation uniquement**, en conservant le même découpage et les mêmes hyperparamètres. Les modèles d'ablation ne sont **pas** comparés sur le test final pour choisir une variante.

Avant un déploiement réel, confirmer que `last_invoice_amount` correspond bien à une facture connue au snapshot ; si elle est calculée après le churn, il faudra l'exclure ainsi que ses dérivées.

In [31]:
invoice_family = [
    "last_invoice_amount",
    "invoice_fee_ratio",
    "invoice_fee_gap",
    "payment_invoice_interaction",
    "invoice_below_fee",
]
variants = {
    "Modèle complet": MODEL_FEATURES,
    "Sans ratio facture": [f for f in MODEL_FEATURES if f != "invoice_fee_ratio"],
    "Sans ratio ni écart": [
        f for f in MODEL_FEATURES if f not in {"invoice_fee_ratio", "invoice_fee_gap"}
    ],
    "Sans famille facture": [f for f in MODEL_FEATURES if f not in set(invoice_family)],
}
ablation_rows = []
for variant_name, features in variants.items():
    if variant_name == "Modèle complet":
        pv = probas["Validation"]
    else:
        require_no_active_run()
        with mlflow.start_run(
            run_name=variant_name, tags=training_tags("diagnostic", training_run_id)
        ) as diagnostic_run:
            ablation_model = CatBoostClassifier(**MODEL_PARAMS, verbose=False)
            diagnostic_frames = {"train": train_df, "validation": val_df}
            diagnostic_contract, diagnostic_datasets = log_training_inputs(
                ablation_model,
                diagnostic_frames,
                features,
                feature_metadata,
                cell_id="train-invoice-ablation",
            )
            mlflow.log_params(
                {"early_stopping_rounds": EARLY_STOPPING_ROUNDS, "use_best_model": True}
            )

            ablation_model.fit(
                make_matrix(train_df, features),
                y_train,
                cat_features=[c for c in CATEGORICAL_FEATURES if c in features],
                eval_set=(make_matrix(val_df, features), y_val),
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                use_best_model=True,
                verbose=False,
                callbacks=[MLflowTrainingCallback()],
            )
            log_diagnostic_model(
                ablation_model, diagnostic_contract, diagnostic_frames, diagnostic_datasets
            )
            ablation_run_id = diagnostic_run.info.run_id
        pv = ablation_model.predict_proba(make_matrix(val_df, features))[:, 1]
    ablation_rows.append(
        {
            "Variante": variant_name,
            "Features": len(features),
            "Average Precision": average_precision_score(y_val, pv),
            "ROC-AUC": roc_auc_score(y_val, pv),
            "Brier": brier_score_loss(y_val, pv),
        }
    )
ablation_results = pd.DataFrame(ablation_rows)
display(ablation_results)
fig = px.bar(
    ablation_results,
    x="Variante",
    y="Average Precision",
    color="Variante",
    text_auto=".3f",
    title="Ablation facture — validation uniquement",
)
fig.update_layout(template="plotly_white", height=430, showlegend=False, xaxis_tickangle=-15)
fig.show(config={"responsive": True})

,Variante,Features,Average Precision,ROC-AUC,Brier
0,Modèle complet,30,0.993330,0.999705,0.001452
1,Sans ratio facture,30,0.993330,0.999705,0.001452
2,Sans ratio ni écart,29,0.985301,0.998899,0.003109
3,Sans famille facture,27,0.129804,0.776320,0.030754


## 9 — Transmission au notebook 03

Le prochain notebook charge ce modèle et ses données dans MLflow, sans dépendre des variables du noyau courant ni des CSV futurs. Il calcule les diagnostics de classification, les SHAP et le bootstrap du test. Les modèles de diagnostic restent consultables séparément dans MLflow.

In [32]:
saved_run = MlflowClient().get_run(training_run_id)
print("Run principal :", training_run_id)
print("Statut :", saved_run.info.status)
print("Modèle :", model_info.model_uri)
print("Suite : 03_churn_evaluation_et_explicabilite.ipynb")

Run principal : 7b12e0c23caa4922b150fbbc69a30bc9
Statut : FINISHED
Modèle : models:/m-e5ce40243b2441efa0c3476f03f34557
Suite : 03_churn_evaluation_et_explicabilite.ipynb
